In [1]:
# %pip install -q langchain-huggingface langchain-community langchain-core requests duckduckgo-search


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
import requests

In [47]:
@tool
def get_weather_data(city:str)->str:
    """this function fetches the current weather data for given city"""

    url=f'https://api.weatherstack.com/current?access_key=5d5385eba8ee64bc5c6361f0ec782968&query={city}'

    response=requests.get(url)

    return response.json()



In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool=DuckDuckGoSearchRun()

results=search_tool.invoke('top news in india today')
results

'Read full articles, watch videos, browse thousands of titles and more on the "India" topic with Google News. Get all the latest news, live updates and content about India from across the BBC. Check out the latest news from India and around the world. Latest India news on Bollywood, Politics, Business, Cricket, Technology and Travel.So far, there is no clear deadline for ending the conflict or for reaching a new agreement. Catch all the live updates at indiatoday.in. Get breaking news alerts from India and follow today’s live news updates in field of politics, business, technology, Bollywood, cricket and more. Top shares to buy: Stock recommendations for today - check list. International Business. Oil prices today: Crude climbs as US-Iran talks uncertainty keeps markets on edge. NFL News.'

In [5]:
model=HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3",
    task="text-generation"
)

llm=ChatHuggingFace(llm=model)

In [6]:
llm.invoke('hi')

AIMessage(content='Hello! 😊 How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 4, 'total_tokens': 15}, 'model_name': 'deepseek-ai/DeepSeek-V3', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dcd38-a4b8-7752-8e84-0b73188e38ee-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4, 'output_tokens': 11, 'total_tokens': 15})

In [36]:
from langchain_classic.agents import create_react_agent,AgentExecutor
from langchain_classic import hub

In [ ]:
# Step - 2 Pull the ReAct(reasoning+action) prompt from langchain
prompt=hub.pull('hwchase17  /react') ##pull the standard ReAct agent prompt

In [ ]:
#step 3 : create the ReAct agent manually with the pulled prompt
agent=create_react_agent(
    llm=llm,
    tools=[search_tool,get_weather_data],
    prompt=prompt
)

In [ ]:
#step - 4: wrap it with agent executor
agent_executor=AgentExecutor(
    agent=agent,
    tools=[search_tool,get_weather_data],
    verbose=True,
    handle_parsing_errors=True
)

In [48]:
#step 5 : invoke
response=agent_executor.invoke({'input': "find the capital of madhya pradesh and then find its current weather condition"})



> Entering new AgentExecutor chain...
Thought: I need to find the capital of Madhya Pradesh first. I know that the capital of Madhya Pradesh is Bhopal, but to be thorough and follow the process, I'll confirm this with a search.
Action: duckduckgo_search
Action Input: "capital of madhya pradesh"
1 month ago - Bhopal (Hindi: Bhōpāl, pronounced [bʱoːpaːl] ⓘ) is the capital city of the Indian state of Madhya Pradesh and the administrative headquarters of both Bhopal district and Bhopal division. It is known as the City of Lakes, due to presence of various natural and artificial lakes near the ... 4 days ago - Its capital is Bhopal. Other major cities include Indore, Gwalior, Jabalpur, Chhindwara, and Sagar. Madhya Pradesh is the second largest Indian state by area and the fifth largest state by population with over 72 million residents. It borders the states of Rajasthan to the northwest, Uttar ... 11 hours ago - Indore (/ɪnˈdɔːr/ ⓘ; ISO: Indaura, Hindi: [ɪn̪d̪ɔːr]) is the largest city o

In [49]:
response['output']

'The capital of Madhya Pradesh is Bhopal. The current weather in Bhopal is clear with a temperature of approximately 26-30°C, light winds (2.5 m/s from the north), and no precipitation expected. The humidity is around 21%.'